In [21]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Normalization

class MockBroker:
    def __init__(self, Balance):
        self.holdings = {}
        self.Balance = Balance

    def __str__(self):
        equity = 0;
        for key, value in self.holdings.items():
            equity += price[key][-1]*value
        return "Equity: " + str(equity) + "\n Holdings: " + str(self.holdings) + "\n Balance: " + str(self.Balance)

    def buy(self, stock, price, qty):
        if self.Balance >= price * int(qty):
            org_qty = int(self.holdings.get(stock, 0))
            self.holdings.update({stock: org_qty + int(qty)})
            self.Balance -= price * int(qty)
            print(f"Bought {qty} of {stock} at {price} each")
            print(self.holdings)
            print("Balance: " + str(self.Balance))
        else:
            print("Not enough balance")

    def sell(self, stock, price, qty):
        org_qty = int(self.holdings.get(stock, 0))
        if org_qty >= int(qty):
            self.holdings.update({stock: org_qty - int(qty)})
            self.Balance += price * int(qty)
            print(f"Sold {qty} of {stock} at {price} each")
            print(self.holdings)
            print("Balance: " + str(self.Balance))
        else:
            print("Can't sell more than bought")

    def sell_off(self, i):
        for key, value in self.holdings.items():
            self.sell(key, price[key][i], value)

    def get_holdings(self):
        return self.holdings

nifty_symbols = [
    "ADANIPORTS", "ASIANPAINT", "AXISBANK", "BAJAJFINSV",
    "BAJFINANCE", "BPCL", "BRITANNIA", "CIPLA", "COALINDIA",
    "DIVISLAB", "DRREDDY", "EICHERMOT", "GRASIM", "HCLTECH",
    "HDFCBANK", "HDFCLIFE", "HEROMOTOCO", "HINDALCO", "HINDUNILVR",
    "ICICIBANK", "ICICIGI", "IOC", "INDUSINDBK", "INFY",
    "ITC", "JSWSTEEL", "KOTAKBANK", "LTTS", "LT",
    "MARICO", "MARUTI", "NESTLEIND"
]

# Initialize the broker
broker = MockBroker(100000)

models = {}
norm_layers = {}

# Load models
for symbol in nifty_symbols:
    models[symbol] = load_model(f'drive/MyDrive/models/{symbol}_model.h5')

# Dictionary to store test data and predictions
data_test = {}
data_pred = {}
price = {}

# Load the test data
# Normalization layer (reusing the one from training)


for symbol in nifty_symbols:
    # Load the test data
    price[symbol] = pd.read_csv(f'drive/MyDrive/data/{symbol}.csv').iloc[-500:, 0].values
    data_test[symbol] = pd.read_csv(f'drive/MyDrive/data/{symbol}.csv').iloc[-500:, 1:5].values

    # Normalize the test data
    norm_l = Normalization(axis=-1)
    norm_l.adapt(data_test[symbol])  # Ensure normalization layer adapts to the test data
    data_test_n = norm_l(data_test[symbol])

    # Reshape the data for LSTM input
    data_test_n = np.reshape(data_test_n, (data_test_n.shape[0], 1, data_test_n.shape[1]))

    # Predict using the model
    data_pred[symbol] = models[symbol].predict(data_test_n)

for i in range(500):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    broker.sell_off(i)

    for symbol, prediction in top_3_symbols[:2]:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (current_price))
        if prediction > 0:
            broker.buy(symbol, current_price, qty)

print(broker)


Streaming output truncated to the last 5000 lines.
Balance: 115505.49652
Sold 0 of HDFCLIFE at 632.75 each
{'EICHERMOT': 0, 'HDFCBANK': 0, 'HEROMOTOCO': 0, 'HDFCLIFE': 0, 'DRREDDY': 0}
Balance: 115505.49652
Sold 0 of DRREDDY at 5823.3501 each
{'EICHERMOT': 0, 'HDFCBANK': 0, 'HEROMOTOCO': 0, 'HDFCLIFE': 0, 'DRREDDY': 0}
Balance: 115505.49652
Bought 34.0 of EICHERMOT at 3338.7 each
{'EICHERMOT': 34, 'HDFCBANK': 0, 'HEROMOTOCO': 0, 'HDFCLIFE': 0, 'DRREDDY': 0}
Balance: 1989.6965200000122
Bought 1.0 of HDFCBANK at 1610.9 each
{'EICHERMOT': 34, 'HDFCBANK': 1, 'HEROMOTOCO': 0, 'HDFCLIFE': 0, 'DRREDDY': 0}
Balance: 378.79652000001215
--- Day 280 ---
Sold 34 of EICHERMOT at 3327.2 each
{'EICHERMOT': 0, 'HDFCBANK': 1, 'HEROMOTOCO': 0, 'HDFCLIFE': 0, 'DRREDDY': 0}
Balance: 113503.59652
Sold 1 of HDFCBANK at 1606.2 each
{'EICHERMOT': 0, 'HDFCBANK': 0, 'HEROMOTOCO': 0, 'HDFCLIFE': 0, 'DRREDDY': 0}
Balance: 115109.79652
Sold 0 of HEROMOTOCO at 2981.75 each
{'EICHERMOT': 0, 'HDFCBANK': 0, 'HEROMOTOC